# TAC-LAnoBERT Phase 4: Main Experiments (E1-E3)

**Purpose**: Run inference and comparison between LAnoBERT baseline and TAC-LAnoBERT.

**Prerequisites** (✅ Already Complete):
- ✅ Phase 2: LAnoBERT baseline trained → `outputs/BGL_lanobert/`
- ✅ Phase 3: TAC-LAnoBERT trained → `outputs/BGL_tac/`

**Experiments**:
- **E1**: Baseline Metrics Verification (load Phase 2 results)
- **E2**: TAC Inference + Comparison (Memory Queue + Hybrid Scoring)
- **E3**: Early Detection Test (measure DLT, EWR)

**GPU Required**: T4 x2 or P100 (for inference only)

**Expected Runtime**: ~3 hours total
- E1: ~5 min (load existing results)
- E2: ~2h (TAC inference with Memory Queue)
- E3: ~1h (DLT analysis)

**Exit Criteria**:
- DLT > 0 (TAC-LAnoBERT shows early warning capability)
- FPR ≤ baseline (no increase in false positives)
- F1/AUROC maintained (detection quality preserved)

## 0. Setup Environment

In [ ]:
# Clone repository
!git clone https://github.com/rubyhcm/TAC-LAnoBERT.git
%cd TAC-LAnoBERT

print("Repository cloned successfully")

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install pytest -q

print("✅ Dependencies installed")

In [ ]:
# Verify PyTorch + CUDA
import torch
import transformers
import os, sys, json
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "❌ GPU not available! Enable GPU in notebook settings."
print("\n✅ Environment ready")

In [ ]:
# Check Phase 2/3 outputs and BGL preprocessed data
import glob

print("="*70)
print("PHASE 4 PREREQUISITES CHECK")
print("="*70)

# Check for Phase 2 & Phase 3 outputs in Kaggle input datasets
baseline_in_input = glob.glob("/kaggle/input/**/BGL_lanobert", recursive=True)
tac_in_input = glob.glob("/kaggle/input/**/BGL_tac", recursive=True)

print("\n1. Checking for Phase 2 baseline (outputs/BGL_lanobert)...")
if os.path.exists("outputs/BGL_lanobert/model/final"):
    print("   ✅ Already present in working directory")
elif baseline_in_input:
    print(f"   📦 Found in Kaggle input: {baseline_in_input[0]}")
    print("   Copying to outputs/...")
    os.makedirs("outputs", exist_ok=True)
    os.system(f"cp -r {baseline_in_input[0]} outputs/")
    print("   ✅ Copied successfully")
else:
    print("   ❌ NOT FOUND")
    print("   Action: Add Phase 2 outputs as Kaggle dataset")
    print("   - Create dataset from local outputs/BGL_lanobert/")
    print("   - Attach to this notebook")
    raise FileNotFoundError("Phase 2 baseline not found. Upload outputs/BGL_lanobert to Kaggle dataset.")

print("\n2. Checking for Phase 3 TAC (outputs/BGL_tac)...")
if os.path.exists("outputs/BGL_tac/model/final"):
    print("   ✅ Already present in working directory")
elif tac_in_input:
    print(f"   📦 Found in Kaggle input: {tac_in_input[0]}")
    print("   Copying to outputs/...")
    os.makedirs("outputs", exist_ok=True)
    os.system(f"cp -r {tac_in_input[0]} outputs/")
    print("   ✅ Copied successfully")
else:
    print("   ❌ NOT FOUND")
    print("   Action: Add Phase 3 outputs as Kaggle dataset")
    print("   - Create dataset from local outputs/BGL_tac/")
    print("   - Attach to this notebook")
    raise FileNotFoundError("Phase 3 TAC model not found. Upload outputs/BGL_tac to Kaggle dataset.")

# ─── BGL Data (preprocessed preferred, raw BGL.log fallback) ─────────────
print("\n3. Checking for BGL preprocessed data (⚡ Recommended - saves ~30 min)...")

# Essential preprocessed files needed for Phase 4 (see PHASE4_README Section 1.3)
REQUIRED_FILES = [
    "data/BGL/BGL_test_parsed.log",
    "data/BGL/BGL_test_label.log",
    "data/BGL/BGL_test_parsed.timestamps",
    "data/BGL/BGL_test.raw",
    "data/BGL/BGL_train_normal_parsed.log",
    "data/BGL/BGL_train_normal_parsed.timestamps",
]

bgl_data_ready = all(os.path.exists(f) for f in REQUIRED_FILES)

if bgl_data_ready:
    print("   ✅ All preprocessed BGL files already present in data/BGL/")
else:
    # Priority 1: Copy from Kaggle input dataset (BGL_data.zip — README Step 1.3)
    # Structure after unzip: /kaggle/input/<dataset-name>/BGL/<files>
    bgl_data_in_input = glob.glob("/kaggle/input/**/BGL/", recursive=True)
    if not bgl_data_in_input:
        # Also try flat structure where BGL files sit directly in dataset root
        bgl_data_in_input = [
            d for d in glob.glob("/kaggle/input/*/")
            if os.path.exists(os.path.join(d, "BGL_test_parsed.log"))
        ]

    if bgl_data_in_input:
        src = bgl_data_in_input[0].rstrip("/")
        print(f"   📦 Found BGL preprocessed data in Kaggle input: {src}")
        print("   Copying to data/BGL/ ... (⚡ saves ~30 min of preprocessing)")
        os.makedirs("data/BGL", exist_ok=True)
        os.system(f"cp -rn {src}/. data/BGL/")
        still_missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
        if not still_missing:
            print("   ✅ Preprocessed BGL data copied successfully")
            bgl_data_ready = True
        else:
            print(f"   ⚠️  Still missing after copy: {still_missing}")

    if not bgl_data_ready:
        # Priority 2: Find raw BGL.log and split + preprocess (~30 min extra)
        print("   ⚠️  Preprocessed BGL data not found. Falling back to raw BGL.log...")
        bgl_logs = glob.glob("/kaggle/input/**/BGL.log", recursive=True)
        if bgl_logs:
            print(f"   📦 Found BGL.log: {bgl_logs[0]}")
            os.makedirs("data/BGL", exist_ok=True)
            os.system(f"ln -sf {bgl_logs[0]} data/BGL/BGL.log")
            print("   Splitting data... (~30 min)")
            !python -m tac_lanobert.split_tac --config configs/bgl_tac_full.yaml
            print("   ✅ Data split complete")
        else:
            print("   ❌ No BGL data found!")
            print("   TIP: Upload preprocessed BGL data (saves 30 min) — see PHASE4_README Step 1.3")
            raise FileNotFoundError(
                "BGL data not found. Either:\n"
                "  (Recommended) Upload BGL_data.zip (data/BGL/) as Kaggle dataset and attach it.\n"
                "  (Fallback) Add BGL.log raw dataset and attach it."
            )

print("\n" + "="*70)
print("✅ ALL PREREQUISITES SATISFIED")
print("="*70)

In [ ]:
# Preprocess test data if not already done
print("\n" + "="*70)
print("PREPROCESSING TEST DATA")
print("="*70)

test_parsed = "data/BGL/BGL_test_parsed.log"
test_label  = "data/BGL/BGL_test_label.log"

if os.path.exists(test_parsed) and os.path.exists(test_label):
    print("\n✅ Test data already preprocessed (from BGL preprocessed dataset — skipping)")
else:
    print("\n⚠️  Test data not preprocessed yet. Running preprocessing...")
    !python -m tac_lanobert.preprocess_tac --config configs/bgl_tac_full.yaml --split test
    print("✅ Test data preprocessed")

# Verify
if os.path.exists(test_parsed) and os.path.exists(test_label):
    with open(test_parsed) as f:
        num_lines = sum(1 for _ in f)
    print(f"   Test lines: {num_lines:,}")
    print("\n✅ Ready for inference")
else:
    raise FileNotFoundError("❌ Failed to preprocess test data")

print("="*70)

## E1: Baseline Metrics Verification

**Goal**: Load and verify Phase 2 baseline metrics as reference.

**Expected Results** (from Phase 2 - 2026-08-24):
- F1: 0.999974 (paper: 1.000, diff: 0.0026%)
- AUROC: 0.999998 (paper: 1.000, diff: 0.0002%)
- FPR: 0.000020 (18 FP / 903,310 normal)
- Best threshold: 7.64127

**Strategy**: Load existing Phase 2 results from `outputs/BGL_lanobert/results/`

In [ ]:
# Verify Phase 2 baseline artifacts exist
baseline_model_dir = "outputs/BGL_lanobert/model/final"
baseline_results_dir = "outputs/BGL_lanobert/results"

baseline_model_exists = (
    os.path.exists(os.path.join(baseline_model_dir, "model.safetensors")) or
    os.path.exists(os.path.join(baseline_model_dir, "pytorch_model.bin"))
)

baseline_scores_exist = os.path.exists(os.path.join(baseline_results_dir, "scores_error_mean.npy"))

print("="*70)
print("E1: BASELINE METRICS VERIFICATION")
print("="*70)
print(f"\nChecking Phase 2 artifacts...\n")
print(f"  {'✅' if baseline_model_exists else '❌'}  Model: {baseline_model_dir}")
print(f"  {'✅' if baseline_scores_exist else '❌'}  Scores: {baseline_results_dir}/scores_*.npy")

if not baseline_model_exists or not baseline_scores_exist:
    print("\n❌ Phase 2 baseline not found!")
    print("   Please upload Phase 2 outputs to Kaggle dataset first.")
    raise FileNotFoundError("Phase 2 baseline artifacts missing")

print("\n✅ Phase 2 baseline artifacts verified")

In [ ]:
# Skip preprocessing - already done in Phase 2
print("⏭️  Skipping baseline preprocessing (Phase 2 complete)")
print("⏭️  Skipping baseline tokenizer training (Phase 2 complete)")
print("⏭️  Skipping baseline model training (Phase 2 complete)")
print("⏭️  Skipping baseline inference (Phase 2 complete)")
print("\n✅ All baseline artifacts already available from Phase 2")

In [ ]:
# Run TAC inference (Memory Queue + Hybrid Scoring)
print("="*70)
print("TAC-LANOBERT INFERENCE (Memory Queue + Hybrid Scoring)")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

!python -m tac_lanobert.inference_tac --config configs/bgl_tac_full.yaml

print(f"\nEnd: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ TAC inference complete")

In [ ]:
# Load and display E1 results
import numpy as np
import re

# Helper to parse text report into dict
def parse_text_report(report_path):
    if not os.path.exists(report_path):
        return None
    with open(report_path, 'r') as f:
        content = f.read()
    metrics = {}
    if m := re.search(r'AUROC:\s+([0-9.e+-]+)', content):
        metrics['auroc'] = float(m.group(1))
    if m := re.search(r'best_F1:\s+([0-9.e+-]+)', content):
        metrics['best_f1'] = float(m.group(1))
    if m := re.search(r'best_threshold:\s+([0-9.e+-]+)', content):
        metrics['best_threshold'] = float(m.group(1))
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics['fpr'] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        metrics['precision'] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        metrics['recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return metrics

results_dir = "outputs/BGL_lanobert/results"

print("="*70)
print("E1 RESULTS — BASELINE REPRODUCTION")
print("="*70)

# Load scores
scores_file = os.path.join(results_dir, "scores_error_mean.npy")
if os.path.exists(scores_file):
    scores = np.load(scores_file)
    print(f"\nScores loaded: {len(scores):,} lines")
    print(f"  Min:  {scores.min():.6f}")
    print(f"  Max:  {scores.max():.6f}")
    print(f"  Mean: {scores.mean():.6f}")
    print(f"  Std:  {scores.std():.6f}")

# Load metrics report (text format)
report_file = os.path.join(results_dir, "BGL_error_mean_report.txt")
report = parse_text_report(report_file)

if report:
    print(f"\n{'='*70}")
    print("BASELINE METRICS (error_mean)")
    print(f"{'='*70}")
    
    def fmt(val):
        return f"{val:.6f}" if isinstance(val, (int, float)) else str(val)
    
    print(f"\n  F1-Score:      {fmt(report.get('best_f1', 'N/A'))}")
    print(f"  Precision:     {fmt(report.get('precision', 'N/A'))}")
    print(f"  Recall:        {fmt(report.get('recall', 'N/A'))}")
    print(f"  AUROC:         {fmt(report.get('auroc', 'N/A'))}")
    print(f"\n  Best Threshold: {fmt(report.get('best_threshold', 'N/A'))}")
    print(f"  FPR:           {fmt(report.get('fpr', 'N/A'))}")
    
    # Compare with Phase 2 expected
    print(f"\n{'='*70}")
    print("PHASE 2 TARGET VALIDATION")
    print(f"{'='*70}")
    
    expected = {
        'best_f1': 0.999974,
        'auroc': 0.999998,
        'fpr': 0.000020,
    }
    
    for metric, target in expected.items():
        actual = report.get(metric)
        if actual is not None:
            diff_pct = abs(actual - target) / target * 100
            status = "✅" if diff_pct < 2.0 else "⚠️"
            print(f"  {status} {metric.upper():10s}  Target={target:.6f}  Actual={actual:.6f}  Δ={diff_pct:.4f}%")
    
    print(f"\n✅ E1 Complete — Baseline metrics confirmed")
    
    # Save E1 reference for E2 comparison
    e1_ref = {
        'experiment': 'E1',
        'model': 'LAnoBERT_baseline',
        'f1': report.get('best_f1'),
        'auroc': report.get('auroc'),
        'fpr': report.get('fpr'),
        'precision': report.get('precision'),
        'recall': report.get('recall'),
        'threshold': report.get('best_threshold'),
        'timestamp': datetime.now().isoformat(),
    }
    
    e1_ref_file = "outputs/phase4_e1_baseline_reference.json"
    with open(e1_ref_file, 'w') as f:
        json.dump(e1_ref, f, indent=2)
    print(f"\n📊 E1 reference saved → {e1_ref_file}")
    
else:
    print(f"\n⚠️  Metrics report not found: {report_file}")
    print("   Check if inference completed successfully.")

## E2: TAC Inference + Main Comparison

**Goal**: Run TAC-LAnoBERT inference and compare against baseline.

**Status**: 
- ✅ TAC-LAnoBERT trained (Phase 3) → `outputs/BGL_tac/model/final/`
- ✅ Timestamps extracted → `data/BGL/*.timestamps`

**What we'll do**:
1. Run TAC inference (Memory Queue + Hybrid Scoring)
2. Generate TAC metrics report
3. Compare with baseline

**Metrics to Compare**:
- F1-Score (window-level)
- AUROC
- FPR (False Positive Rate)
- PR-AUC

**Hypothesis**:
- H1: Time2Vec reduces FPR by ≥15% (compared to baseline)
- H2: Session Memory enables DLT > 0 (early warning capability)

In [ ]:
# Verify Phase 3 TAC artifacts exist
tac_model_dir = "outputs/BGL_tac/model/final"
tac_results_dir = "outputs/BGL_tac/results"

tac_model_exists = (
    os.path.exists(os.path.join(tac_model_dir, "model.safetensors")) or
    os.path.exists(os.path.join(tac_model_dir, "pytorch_model.bin"))
)

tac_tokenizer_exists = os.path.exists("outputs/BGL_tac/tokenizer/BGL_LogBERT-vocab.txt")
tac_timestamps_exist = (
    os.path.exists("data/BGL/BGL_train_normal_parsed.timestamps") and
    os.path.exists("data/BGL/BGL_test_parsed.timestamps")
)

print("="*70)
print("E2: TAC-LANOBERT INFERENCE + COMPARISON")
print("="*70)
print(f"\nChecking Phase 3 artifacts...\n")
print(f"  {'✅' if tac_model_exists else '❌'}  Model: {tac_model_dir}")
print(f"  {'✅' if tac_tokenizer_exists else '❌'}  Tokenizer: outputs/BGL_tac/tokenizer/")
print(f"  {'✅' if tac_timestamps_exist else '❌'}  Timestamps: data/BGL/*.timestamps")

if not tac_model_exists:
    print("\n❌ Phase 3 TAC model not found!")
    print("   Please upload Phase 3 outputs to Kaggle dataset first.")
    raise FileNotFoundError("Phase 3 TAC model missing")

if not tac_tokenizer_exists or not tac_timestamps_exist:
    print("\n⚠️  Some TAC artifacts missing, but can be regenerated.")

print("\n✅ Phase 3 TAC model verified")

In [ ]:
# Regenerate timestamps if missing (quick operation)
if not tac_timestamps_exist:
    print("Regenerating timestamps...\n")
    
    if not os.path.exists("data/BGL/BGL_train_normal_parsed.timestamps"):
        print("Extracting training timestamps...")
        !python -m tac_lanobert.preprocess_tac --config configs/bgl_tac_full.yaml --split train --extract_timestamps
    
    if not os.path.exists("data/BGL/BGL_test_parsed.timestamps"):
        print("Extracting test timestamps...")
        !python -m tac_lanobert.preprocess_tac --config configs/bgl_tac_full.yaml --split test --extract_timestamps
    
    print("\n✅ Timestamps regenerated")
else:
    print("⏭️  Timestamps already exist")

# Skip tokenizer regeneration if exists
if not tac_tokenizer_exists:
    print("\nRegenerating tokenizer...")
    !python -m tac_lanobert.tokenizer_tac --config configs/bgl_tac_full.yaml
    print("✅ Tokenizer regenerated")
else:
    print("⏭️  Tokenizer already exists")

print("\n⏭️  Skipping TAC model training (Phase 3 complete)")

In [ ]:
# Run TAC inference (Memory Queue + Hybrid Scoring)
print("="*70)
print("TAC-LANOBERT INFERENCE (Memory Queue + Hybrid Scoring)")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

!python -m tac_lanobert.inference_tac --config configs/bgl_tac_full.yaml

print(f"\nEnd: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ TAC inference complete")

In [ ]:
# Load and compare E2 results
print("="*70)
print("E2 RESULTS — TAC-LANOBERT vs BASELINE")
print("="*70)

tac_results_dir = "outputs/BGL_tac/results"

# Load TAC scores
tac_mlm_scores = os.path.join(tac_results_dir, "scores_tac_mlm_error.npy")
tac_maha_scores = os.path.join(tac_results_dir, "scores_tac_mahalanobis.npy")
tac_hybrid_scores = os.path.join(tac_results_dir, "scores_tac_hybrid.npy")

if os.path.exists(tac_hybrid_scores):
    scores_mlm = np.load(tac_mlm_scores)
    scores_maha = np.load(tac_maha_scores)
    scores_hybrid = np.load(tac_hybrid_scores)
    
    print(f"\nScores loaded: {len(scores_hybrid):,} lines\n")
    
    print("MLM Loss (reactive):")
    print(f"  Min:  {scores_mlm.min():.6f}")
    print(f"  Max:  {scores_mlm.max():.6f}")
    print(f"  Mean: {scores_mlm.mean():.6f}")
    
    print("\nMahalanobis Distance (proactive):")
    print(f"  Min:  {scores_maha.min():.6f}")
    print(f"  Max:  {scores_maha.max():.6f}")
    print(f"  Mean: {scores_maha.mean():.6f}")
    
    print("\nHybrid Score (α=0.5):")
    print(f"  Min:  {scores_hybrid.min():.6f}")
    print(f"  Max:  {scores_hybrid.max():.6f}")
    print(f"  Mean: {scores_hybrid.mean():.6f}")
    
    # Load TAC metrics report (text format)
    tac_report_file = os.path.join(tac_results_dir, "BGL_tac_hybrid_report.txt")
    tac_report = parse_text_report(tac_report_file)
    
    if tac_report:
        # Load E1 baseline reference
        e1_ref_file = "outputs/phase4_e1_baseline_reference.json"
        if os.path.exists(e1_ref_file):
            with open(e1_ref_file, 'r') as f:
                e1_ref = json.load(f)
        else:
            e1_ref = {'f1': None, 'auroc': None, 'fpr': None}
        
        print(f"\n{'='*70}")
        print("COMPARISON — BASELINE vs TAC-LANOBERT")
        print(f"{'='*70}\n")
        
        
        metrics = ['best_f1', 'auroc', 'fpr', 'precision', 'recall']
        
        print(f"{'Metric':<15} {'Baseline':<15} {'TAC-LAnoBERT':<15} {'Δ':<15} {'Status'}")
        print("-" * 70)
        
        for metric in metrics:
            baseline_val = e1_ref.get(metric if metric != 'best_f1' else 'f1')
            tac_val = tac_report.get(metric)
            
            if baseline_val is not None and tac_val is not None:
                delta = tac_val - baseline_val
                delta_pct = (delta / baseline_val * 100) if baseline_val != 0 else 0
                
                # Determine status
                if metric == 'fpr':
                    # Lower is better
                    status = "✅ Better" if delta < 0 else ("⚠️ Worse" if delta > 0 else "≈ Same")
                    improvement = f"{delta_pct:+.2f}%" if delta != 0 else "0.00%"
                else:
                    # Higher is better
                    status = "✅ Better" if delta > 0 else ("⚠️ Worse" if delta < 0 else "≈ Same")
                    improvement = f"{delta_pct:+.2f}%" if delta != 0 else "0.00%"
                
                display_name = 'F1' if metric == 'best_f1' else metric.upper()
                print(f"{display_name:<15} {baseline_val:<15.6f} {tac_val:<15.6f} {improvement:<15} {status}")
            else:
                print(f"{metric.upper():<15} {'N/A':<15} {'N/A':<15} {'N/A':<15} {'N/A'}")
        
        print(f"\n{'='*70}")
        print("HYPOTHESIS TESTING")
        print(f"{'='*70}\n")
        
        # H1: FPR reduction ≥15%
        if e1_ref.get('fpr') and tac_report.get('fpr'):
            fpr_reduction = (e1_ref['fpr'] - tac_report['fpr']) / e1_ref['fpr'] * 100
            h1_status = "✅ PASS" if fpr_reduction >= 15 else "❌ FAIL"
            print(f"H1: FPR reduction ≥15%")
            print(f"    Actual reduction: {fpr_reduction:.2f}%  {h1_status}")
        
        # H2: DLT > 0 (will be measured in E3)
        print(f"\nH2: DLT > 0 (Early Warning)")
        print(f"    → Deferred to E3: Early Detection Test")
        
        # Save E2 comparison report
        e2_report = {
            'experiment': 'E2',
            'timestamp': datetime.now().isoformat(),
            'baseline': e1_ref,
            'tac_lanobert': {
                'f1': tac_report.get('best_f1'),
                'auroc': tac_report.get('auroc'),
                'fpr': tac_report.get('fpr'),
                'precision': tac_report.get('precision'),
                'recall': tac_report.get('recall'),
                'threshold': tac_report.get('best_threshold'),
            },
            'improvements': {
                'fpr_reduction_pct': fpr_reduction if e1_ref.get('fpr') and tac_best.get('fpr') else None,
            },
        }
        
        e2_report_file = "outputs/phase4_e2_comparison_report.json"
        with open(e2_report_file, 'w') as f:
            json.dump(e2_report, f, indent=2)
        
        print(f"\n📊 E2 comparison report saved → {e2_report_file}")
        print("\n✅ E2 Complete")
        
    else:
        print(f"\n⚠️  TAC metrics report not found: {tac_report_file}")
        print("   Check if TAC inference completed successfully.")
else:
    print(f"\n⚠️  TAC scores not found: {tac_hybrid_scores}")
    print("   Check if TAC inference completed successfully.")

## E3: Early Detection Test

**Goal**: Measure Detection Lead Time (DLT) and Early Warning Rate (EWR).

**Metrics**:
- **DLT**: `t_failure - t_first_alert` (in minutes/seconds)
- **EWR**: Percentage of failures with DLT ≥ 5 minutes

**Method**:
1. Load TAC hybrid scores and timestamps
2. Load failure labels and timestamps
3. For each failure, find first alert before it
4. Calculate DLT distribution
5. Report statistics

**Exit Criteria**: DLT > 0 (TAC-LAnoBERT shows early warning capability)

In [ ]:
print("="*70)
print("E3: EARLY DETECTION TEST")
print("="*70)

# Load test data
test_raw = "data/BGL/BGL_test.raw"
test_timestamps = "data/BGL/BGL_test_parsed.timestamps"
tac_hybrid_scores = "outputs/BGL_tac/results/scores_tac_hybrid.npy"

print("\nLoading test data...")

# Load labels (- = normal, anomaly keywords)
labels = []
with open(test_raw, 'r') as f:
    for line in f:
        # BGL format: label is FIRST token (- = normal, anything else = anomaly)
        parts = line.strip().split()
        if len(parts) >= 1:
            label = 0 if parts[0] == '-' else 1  # 0=normal, 1=anomaly
            labels.append(label)

labels = np.array(labels)
print(f"  Labels: {len(labels):,} lines ({labels.sum():,} anomalies)")

# Load timestamps
timestamps = []
with open(test_timestamps, 'r') as f:
    for line in f:
        timestamps.append(float(line.strip()))

timestamps = np.array(timestamps)
print(f"  Timestamps: {len(timestamps):,} lines")

# Load TAC scores
scores = np.load(tac_hybrid_scores)
print(f"  Scores: {len(scores):,} lines")

# Verify alignment
assert len(labels) == len(timestamps) == len(scores), "Data length mismatch!"
print("\n✅ Data loaded and aligned")

In [ ]:
# Determine alert threshold (use best threshold from E2)
tac_report_file = "outputs/BGL_tac/results/BGL_tac_hybrid_report.txt"
tac_report = parse_text_report(tac_report_file)

threshold = None
if tac_report:
    threshold = tac_report.get('best_threshold')

# Fallback: use 99th percentile of normal scores
if threshold is None:
    if (labels == 0).sum() > 0:
        threshold = np.percentile(scores[labels == 0], 99)
        print(f"⚠️  TAC report not found. Using 99th percentile of normal scores: {threshold:.6f}")
    else:
        # Emergency fallback: use 95th percentile of all scores
        threshold = np.percentile(scores, 95)
        print(f"⚠️  No normal samples found. Using 95th percentile of all scores: {threshold:.6f}")
else:
    print(f"✅ Using threshold from TAC report: {threshold:.6f}")

print(f"\nAlert threshold: {threshold:.6f}")

# Generate alerts
alerts = (scores > threshold).astype(int)
print(f"Total alerts: {alerts.sum():,} / {len(alerts):,} lines ({alerts.sum()/len(alerts)*100:.2f}%)")

In [ ]:
# Calculate DLT for each failure
print("\nCalculating Detection Lead Time...\n")

failures = np.where(labels == 1)[0]
dlts = []  # Detection Lead Times (in seconds)

for fail_idx in failures:
    fail_time = timestamps[fail_idx]
    
    # Find first alert before this failure
    # Look back up to 1 hour (3600 seconds)
    lookback_window = 3600  # seconds
    
    # Find indices in lookback window
    mask = (timestamps < fail_time) & (timestamps >= fail_time - lookback_window)
    candidate_indices = np.where(mask & (alerts == 1))[0]
    
    if len(candidate_indices) > 0:
        # Find first alert (earliest time)
        first_alert_idx = candidate_indices[np.argmin(timestamps[candidate_indices])]
        alert_time = timestamps[first_alert_idx]
        dlt = fail_time - alert_time  # seconds
        dlts.append(dlt)
    else:
        # No alert found (DLT = 0 or negative, meaning reactive detection)
        dlts.append(0.0)

dlts = np.array(dlts)

print(f"Total failures analyzed: {len(failures):,}")
print(f"\nDLT Statistics:")
print(f"  Mean DLT:     {dlts.mean():.2f} seconds ({dlts.mean()/60:.2f} minutes)")
print(f"  Median DLT:   {np.median(dlts):.2f} seconds ({np.median(dlts)/60:.2f} minutes)")
print(f"  Max DLT:      {dlts.max():.2f} seconds ({dlts.max()/60:.2f} minutes)")
print(f"  Std DLT:      {dlts.std():.2f} seconds")

# Early Warning Rate (EWR): % failures with DLT ≥ 5 minutes
early_warning_threshold = 5 * 60  # 5 minutes in seconds
ewr = (dlts >= early_warning_threshold).sum() / len(dlts) * 100

print(f"\nEarly Warning Rate (DLT ≥ 5 min): {ewr:.2f}%")

# DLT > 0 rate (any early detection)
dlt_positive_rate = (dlts > 0).sum() / len(dlts) * 100
print(f"DLT > 0 rate (any early detection):  {dlt_positive_rate:.2f}%")

# Histogram
print(f"\nDLT Distribution (buckets in minutes):")
bins = [0, 1, 5, 10, 30, 60]  # minutes
dlts_minutes = dlts / 60
for i in range(len(bins) - 1):
    count = ((dlts_minutes >= bins[i]) & (dlts_minutes < bins[i+1])).sum()
    pct = count / len(dlts) * 100
    print(f"  [{bins[i]:3d}-{bins[i+1]:3d} min): {count:5d} ({pct:5.2f}%)")
count = (dlts_minutes >= bins[-1]).sum()
pct = count / len(dlts) * 100
print(f"  [≥{bins[-1]:2d} min):      {count:5d} ({pct:5.2f}%)")

In [ ]:
# Test Hypothesis H2: DLT > 0
print("="*70)
print("HYPOTHESIS H2: DLT > 0 (Early Warning Capability)")
print("="*70)

h2_status = "✅ PASS" if dlts.mean() > 0 else "❌ FAIL"
print(f"\nMean DLT: {dlts.mean():.2f} seconds  {h2_status}")

if dlts.mean() > 0:
    print("\n✅ TAC-LAnoBERT demonstrates early warning capability!")
    print(f"   Average lead time: {dlts.mean()/60:.2f} minutes before failure")
else:
    print("\n⚠️  No early warning detected. TAC behaves reactively like baseline.")
    print("   Possible causes:")
    print("   - Memory Queue not capturing long-range patterns")
    print("   - Mahalanobis distance not sensitive enough")
    print("   - Hybrid scoring alpha needs tuning")

In [ ]:
# Save E3 report
e3_report = {
    'experiment': 'E3',
    'timestamp': datetime.now().isoformat(),
    'total_failures': int(len(failures)),
    'dlt_statistics': {
        'mean_seconds': float(dlts.mean()),
        'mean_minutes': float(dlts.mean() / 60),
        'median_seconds': float(np.median(dlts)),
        'median_minutes': float(np.median(dlts) / 60),
        'max_seconds': float(dlts.max()),
        'max_minutes': float(dlts.max() / 60),
        'std_seconds': float(dlts.std()),
    },
    'early_warning_rate_5min': float(ewr),
    'dlt_positive_rate': float(dlt_positive_rate),
    'hypothesis_h2': {
        'statement': 'DLT > 0 (Early Warning Capability)',
        'result': 'PASS' if dlts.mean() > 0 else 'FAIL',
        'mean_dlt': float(dlts.mean()),
    },
}

e3_report_file = "outputs/phase4_e3_early_detection_report.json"
with open(e3_report_file, 'w') as f:
    json.dump(e3_report, f, indent=2)

print(f"\n📊 E3 report saved → {e3_report_file}")
print("\n✅ E3 Complete — Early Detection Test")

## Phase 4 Summary

**Experiments Completed**:
- ✅ E1: Baseline Metrics Verification (loaded Phase 2 results)
- ✅ E2: TAC Inference + Comparison
- ✅ E3: Early Detection Test (DLT, EWR)

**Key Findings** (filled after running):
- F1-Score: Baseline vs TAC
- FPR: Baseline vs TAC (H1: ≥15% reduction?)
- DLT: Mean, Median, EWR (H2: DLT > 0?)

**Exit Criteria Check**:
1. DLT > 0? (Early warning capability)
2. FPR ≤ baseline? (No increase in false positives)
3. F1/AUROC maintained? (Detection quality preserved)

**Reports Generated**:
- `outputs/phase4_e1_baseline_reference.json`
- `outputs/phase4_e2_comparison_report.json`
- `outputs/phase4_e3_early_detection_report.json`
- `outputs/phase4_completion_report.json`

**Next Steps**:
- **Phase 5**: Ablation Study (E4: time_only, memory_only, full)
- **Phase 5**: Robustness Tests (E5: workload spike, E6: efficiency, E7: cross-system)
- **Phase 6**: Statistical Analysis (5 runs × seeds, Wilcoxon test, Cohen's d)

In [ ]:
# Generate final Phase 4 summary
print("="*70)
print("PHASE 4 COMPLETION SUMMARY")
print("="*70)

# Load all reports
reports = {}
for exp in ['e1', 'e2', 'e3']:
    report_files = glob.glob(f"outputs/phase4_{exp}_*.json")
    if report_files:
        with open(report_files[0], 'r') as f:
            reports[exp] = json.load(f)

if 'e1' in reports and 'e2' in reports:
    print("\n📊 E1-E2: Metric Comparison")
    print("-" * 70)
    
    e1 = reports['e1']
    e2 = reports['e2'].get('tac_lanobert', {})
    
    for metric in ['f1', 'auroc', 'fpr']:
        baseline = e1.get(metric)
        tac = e2.get(metric)
        if baseline and tac:
            delta = tac - baseline
            delta_pct = (delta / baseline * 100) if baseline != 0 else 0
            print(f"  {metric.upper():10s}  Baseline={baseline:.6f}  TAC={tac:.6f}  Δ={delta_pct:+.2f}%")

if 'e3' in reports:
    print("\n📊 E3: Early Detection")
    print("-" * 70)
    
    e3 = reports['e3']
    dlt = e3.get('dlt_statistics', {})
    print(f"  Mean DLT:  {dlt.get('mean_minutes', 0):.2f} minutes")
    print(f"  EWR (≥5m): {e3.get('early_warning_rate_5min', 0):.2f}%")
    print(f"  DLT > 0:   {e3.get('dlt_positive_rate', 0):.2f}%")

print("\n" + "="*70)
print("EXIT CRITERIA")
print("="*70)

checks = {
    '1. E1 Baseline confirmed': 'e1' in reports,
    '2. E2 TAC-LAnoBERT trained': 'e2' in reports,
    '3. E3 DLT measured': 'e3' in reports,
    '4. DLT > 0 (H2)': reports.get('e3', {}).get('hypothesis_h2', {}).get('result') == 'PASS' if 'e3' in reports else False,
}

for criterion, passed in checks.items():
    print(f"  {'✅' if passed else '❌'}  {criterion}")

if all(checks.values()):
    print("\n🎉 PHASE 4 COMPLETE — Ready for Phase 5: Ablation & Robustness")
else:
    failed = [k for k, v in checks.items() if not v]
    print(f"\n⚠️  Some checks failed: {failed}")

# Save consolidated report
phase4_report = {
    'phase': 4,
    'status': 'complete' if all(checks.values()) else 'partial',
    'completion_date': datetime.now().isoformat(),
    'experiments': reports,
    'exit_criteria': checks,
}

phase4_report_file = "outputs/phase4_completion_report.json"
with open(phase4_report_file, 'w') as f:
    json.dump(phase4_report, f, indent=2)

print(f"\n✅ Phase 4 report saved → {phase4_report_file}")